# 02 · Backtest, score, and explain

Reads `gold_lead_features`, runs a rolling monthly backtest, scores the latest months with reason codes, and writes the results back to the lakehouse for Power BI.

Outputs: `gold_model_backtest`, `gold_model_summary`, `gold_model_calibration`, `gold_lead_scores`, `gold_model_lift`. Experiments are logged to the Fabric MLflow experiment `lead-prioritization`.

In [ ]:
import pandas as pd
from leadscore import lakehouse as lh, modeling as md

spark = lh.get_spark()
gold = lh.read_table(spark, 'gold_lead_features').toPandas()
gold['call_month_start'] = pd.to_datetime(gold['call_month_start'])

## Rolling monthly backtest
Every month is scored by a model trained only on earlier months. The champion (logistic regression) is compared with LightGBM, a model trained once and never retrained, and the seller's rule of thumb.

In [ ]:
bt = md.run_backtest(gold)
display(bt.summary)

In [ ]:
display(md.calibration_table(bt.predictions))

## Score the latest months with reason codes
Reasons are measured against the other leads in the same month, so they explain why a lead ranks above its peers.

In [ ]:
scored = pd.concat([md.score_month(gold, pd.Timestamp(m)) for m in ['2010-09-01', '2010-10-01', '2010-11-01']])
display(scored.head(20))
display(md.lift_table(scored))

In [ ]:
# Persist for Power BI (Direct Lake)
for name, df in {'gold_model_summary': bt.summary, 'gold_lead_scores': scored}.items():
    lh.write_table(spark.createDataFrame(df.assign(**{c: df[c].dt.date for c in df.columns if pd.api.types.is_datetime64_any_dtype(df[c])})), name)